# TP3 : Algorithme Bayésien Naïf

### Identification de l'étudiant

In [ ]:
# Nom : 
# Prénom : 
# Numéro d'étudiant : 

### Modalités de rendu

1. Chaque étudiant doit rendre un travail individuel.
2. Renommez ce fichier selon la convention : `TP3_Nom_Prenom.ipynb`.
3. Le rendu s'effectuera via le lien de dépôt communiqué par votre enseignant.
4. Assurez-vous que votre code s'exécute sans erreur (Menu : Kernel > Restart & Run All).

### Objectifs de la séance

Ce TP a pour objectif l’implémentation d'un algorithme Bayésien Naïf que l'on utilisera pour faire une analyse de sentiments sur des critiques de film.

Les performances des modèles seront évaluées et comparées à l’aide de métriques standards, notamment :
- la matrice de confusion,
- la précision (precision) et rappel (recall).
- ROC, AUC

### Documentation utile
- [Numpy User Guide](https://numpy.org/doc/stable/user/index.html)
- [Scikit-Learn OpenML](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_openml.html)



## Organisation du TP

Ce TP contient 2 parties :

1. **Tokeniser, Lemmatiser et Vectoriser un ensemble de phrases**
2. **Implémentation et Comparaison d'algorithmes Bayésiens Naïfs**

# Partie 1 — Tokeniser, Lemmatiser et Vectoriser un ensemble de phrases

## Importation des modules

In [ ]:
import numpy as np
import pandas as pd
import scipy as sp
import matplotlib.pyplot as plt
import wordcloud

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay, precision_score, recall_score, confusion_matrix

In [ ]:
from pprint import pprint

## Téléchargement des ressources pour NLTK

In [ ]:
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

## Concepts de base

Considérons quelques critiques simples:

In [ ]:
reviews = [
    "I loved every second of this masterpiece!",
    "The actors were terrible and the story was boring.",
    "Amazing music and great special effects!",
    "I would rather watch paint dry than see this movie.",
    "A complete waste of time and money!",
    "Don't miss this masterpiece!"
]

La première étapde en général consiste à mettre tout en lettres minuscules:

In [ ]:
# Complétez les points de suspension
# Utilisez la méthode lower() pour mettre toutes les reviews en minuscule

reviews = [review....() for review in reviews]
pprint(reviews)

Ensuite on découpe la phrase en **tokens**: c'est la tokenisation.

On ne souhaite pas garder les signes de ponctuation:

In [ ]:
def has_letter_digit(word):
    # Retourne vrai si le mot contient au moins un caractère alphanumérique
    for item in word:
        if item.isalnum():
            return True
    return False

# Utilisez la méthode word_tokenize() pour découper la phrase en token
# en excluant ceux qui n'ont pas de caractère alphanumérique
for review in reviews:
    tokens = [word for word in nltk....(review) if ...]
    print(tokens)

Il y a plein de tokens qui n'apportent pas de sens, ce sont des mots vides, ou encore stop-words.

Recommençont l'opération en excluant cette fois les stop-words :

In [ ]:
stop_words = set(stopwords.words('english'))
negations = {'not', 'nor', 'neither'}  # on garde ces tokens parce qu'ils peuvent influencer le sentiment
stop_words -= negations
#quelques exemples de stop word
print(list(stop_words)[0:10])

# On recommence l'opération précédente : 
# Utilisez la méthode word_tokenize() pour découper la phrase en token
# en excluant ceux qui n'ont pas de caractère alphanumérique et que mes mots ne sont pas notre set stop_word.
for review in reviews:
    tokens = [word for word in nltk....(review) if ... and ... not in ...]
    print(tokens)

Enfin, on veut lemmatiser pour simplifier encore. La lemmatisation consiste par exemple à remplacer les pluriels par des singuliers ou à remplacer un verbe conjugué par un verbe à l'infinitif:

Recommençont l'opération en lemmatisant au préalable chaque mot des différentes reviews :

In [ ]:
# Utilisez la méthode lemmatize() de la classe WordNetLemmatizer() pour lemmatiser chaque token

for review in reviews:
    wnl = WordNetLemmatizer()
    tokens = [wnl....(word) for word in nltk....(review) if ... and ... not in ...]
    print(tokens)

La lemmatisation peut ne pas fonctionner correctement. Par exemple *loved* peut être un adjectif ou un verbe. Pour afiner le traitement, on peut avoir besoin de détecter la structure grammaticale de la phrase, ce qui est appelé Part Of Speech (POS):

- **Vxx**: verbe
- **Jxx**: adjectif
- **Nxx**: nom
- **Pxx**: pronom
- **Rxx**: adverbe
- **.**: ponctuation
- **DT**: déterminent
- **CC**: conjonction de coordination
- **IN**: conjonction ou préposition
- **MD**: modal

In [ ]:
# La méthode pos_tag() détecte la structure grammaticale
for review in reviews:
    tokens = nltk.word_tokenize(review)
    pos = nltk.pos_tag(tokens)
    print(pos)

Quand on fait la lemmatisation, on peut préciser le POS afin d'afiner le traitement:

In [ ]:
for review in reviews:
    wnl = WordNetLemmatizer()
    tokens = nltk.word_tokenize(review)
    pos_tokens = nltk....(tokens)
    dct = {'J': wordnet.ADJ, 'V': wordnet.VERB, 'N': wordnet.NOUN, 'R': wordnet.ADV}
    out = []
    for token, pos in zip(tokens, pos_tokens):
        if token in stop_words or pos[1][0] in ('.', 'D', 'M'):
            # On ignore les tokens dans les stop-words, les signes de ponctuations et les déterminents
            continue
        out.append(wnl.lemmatize(token, dct[pos[1][0]]))
    print(out)

*loved* a été simplifié en *love*

## Vectorisation

Dans cette section nous verrons comment on peut vectoriser une critique de film. Dans un premier temps, nous allons créer une fonction de tokenisation personnalisée pour sklearn:

In [ ]:
def has_letter_digit(word):
    # Retourne vrai si le mot contient au moins un caractère alphanumérique
    for item in word:
        if item.isalnum():
            return True
    return False

def has_symbol(word):
    # Retourne vrai si le mot contient un symbole
    for item in word:
        if item in ['/', '\\', '+', '=', '_', '.', '$', '£']:
            return True
    return False
    
def tokenizer():
    stop_words = set(stopwords.words('english'))
    negations = {'not', 'nor', 'neither'}  # on garde ces tokens parce qu'ils peuvent influencer le sentiment
    stop_words -= negations
    wnl = WordNetLemmatizer()
    dct = {'J': wordnet.ADJ, 'V': wordnet.VERB, 'N': wordnet.NOUN, 'R': wordnet.ADV}
    pos_removed = {'.', ',', ':', ';', '`', "'", '$', 'C', 'D', 'M', 'I', 'P', 'W', 'U', 'F', 'S', 'E', 'T', 'L'}
    def inner(review):
        tokens = nltk.word_tokenize(review)
        pos_tokens = nltk.pos_tag(tokens)
        out = []
        for token, pos in zip(tokens, pos_tokens):
            pos0 = pos[1][0]
            if token in stop_words or pos0 in pos_removed:
               # On ignore les token dans les stop-words et certains POS
               continue
            if has_letter_digit(token) and not has_symbol(token):
                out.append(wnl.lemmatize(token, dct[pos0]))
        return out
    return inner

Maintenant que nous avons de quoi tokeniser et lématiser les critiques, On doit convertir ces données en données numériques pour qu'elles puissent être traitées par un algorithme d'apprentissage automatique. Cette opération peut être réalisée par une opération simple de **vectorisation par comptage**.

In [ ]:
# Entraîner un CountVectorizer (stop_words=None, tokenizer=tokenizer(), token_pattern=None) sur reviews

cnt = ...
cnt.fit(...);

In [ ]:
features = cnt.get_feature_names_out()
features

Il est également possible de former des n-grammes afin de prendre en compte quelques interactions de mots. Les n-grammes sont des interactions de mots.
Par exemple dans la phrase: *il fait très beau*, la tokenisation pourrait donner par exemple `['fait', 'très', 'beau']`. Ce sont des 1-grammes. Les 2-grammes seraient: `['fait très', 'très beau']`. Et il y aurait un seul 3-gramme: `['fait très beau']`.

Dans le code ci-dessous on extrait les 1-grammes et les 2-grammes:

In [ ]:
# Pour extraire les 1-gralles jusqu'au n-grammes, il faut ajouter en paramètre ngram_range=(1,n).
# Ici, on veut n=2

cnt = CountVectorizer(stop_words=None, tokenizer=tokenizer(), token_pattern=None, ngram_range=...)
cnt.fit(reviews);

In [ ]:
features = cnt.get_feature_names_out()
features

# Partie 2 — Implémentation et Comparaison d'algorithmes Bayésiens Naïfs

## Application à un dataset réel IMDb

Nous allons maintenant appliquer ça à une base de données de critiques de films publiées sur le site IMDb. Le dataset est publique mais il a été téléchargé depuis ce lien sur Kaggle:

https://www.kaggle.com/datasets/yasserh/imdb-movie-ratings-sentiment-analysis

### Exploration des données

In [ ]:
# Mettre le lien l'emplacement du fichier télécharger en argument

df = pd.read_csv("...")

Utilisez la méthode `info()` pour inspecter le type des variables (float, int, category) et repérer le nombre de valeurs non-nulles.

In [ ]:
df....()

Nous avons deux colonnes. Une première qui contient le texte de la critique et une seconde avec une variable discrète: 1: revue positive, 0: revue négative. 

Utilisez la méthode `value_counts()` pour regarder les effectifs de chaque classe.

In [ ]:
df["label"]....()

Le dataset est équilibré entre les revues positives et les revues négatives.

Visualisation de quelques revues courtes:

In [ ]:
mask = df["text"].str.len() <= 100
for l in df.loc[mask, "text"]:
    print(l)
    print("----")

On peut maintenant tester notre méthode pour transformer la donnée:

In [ ]:
df_short = df[mask].copy()

In [ ]:
# On ne prend en compte que les 1-gramme pour le moment
cnt = ...
cnt.fit(df_short["text"]);

In [ ]:
# On regarde comment certaines des reviews ont été transformé

features = cnt.get_feature_names_out()
array = cnt.transform(df_short["text"]).toarray()

for i, row in enumerate(df_short.iterrows()):
    print(row[1]["text"])
    print(" => ", features[array[i].astype(bool)])

### Modèle avec 1000 données

On ne gardera qu'un échantillon de 1000 revues pour l'apprentissage et le test du modèle.

In [ ]:
df_sample = df.sample(n=..., random_state=14)

In [ ]:
# On veut maintenant séparer notre dataset en 2, un set d'entrainement et un set de test
# On utilise la méthode train_test_split() et on veut que la taille du test de test soit 20% du dataset original.

df_train, df_test = train_test_split(df_sample, test_size=...)

In [ ]:
cnt = ...
cnt.fit(df_train["text"]);

In [ ]:
features = cnt.get_feature_names_out()
x_train = cnt.transform(df_train["text"])  # on garde maintenant le tableau creux !
y_train = df_train["label"].to_numpy()

On créé l'algorithme Bayésien Naïf. On lui demande explicitement de ne pas apprendre de probabilité a priori à partir des données et on lui donne explicitement la probabilité a priori. Si on ne fait pas ça, l'algorithme calcule la probabilité a priori à partir de la répartition des critiques positives et négatives dans les données d'apprentissage:

In [ ]:
# Entraîner un MultinomialNB sur notre set d'entrainement.
# On voudra le régler pour ne pas apprendre de probabilité à priori (indice : utiliser les arguments de la méthode)
# On voudra également fixer la probabilité à priori de chaque classe (indice : idem ). 
# [0.5,0.5] convient mais vous pouver essayer d'autres valeurs

clf = ...
clf.fit(...,...);

In [ ]:
# On regarde les prédictions de notre modèle sur notre set de test.
# On utilise pour ça la méthode predict_proba()

x_test = cnt.transform(df_test["text"])
y_test = df_test["label"].to_numpy()
p_hat = clf.predict_proba(...)[:,1]  # on ne garde que les probabilités de la classe "positive"

Cette fonction de sklearn permet de tracer la courbe ROC à partir des probabilités prédites:

In [ ]:
# On veut utiliser une la méthode from_predictions() de la classe RocCurveDisplay.
# Mettez les bons arguments dans la méthode

RocCurveDisplay....(...);

Et cette fonction permet d'afficher la matrice de confusion pour un seuil de décision à 0.5:

In [ ]:
# On veut utiliser une la méthode from_predictions() de la classe ConfusionMatrixDisplay.

ConfusionMatrixDisplay....(y_test, np.where(p_hat>=0.5, 1, 0));

### Modèle avec 1000 données et des n-grammes

On ne gardera qu'un échantillon de 1000 revues pour l'apprentissage et le test du modèle, mais on rajoute les 2-grammes ;)

In [ ]:
df_sample = df.sample(..., random_state=14)

In [ ]:
# On veut maintenant séparer notre dataset en 2, un set d'entrainement et un set de test
# On utilise la méthode train_test_split() et on veut que la taille du test de test soit 20% du dataset original.

df_train, df_test = train_test_split(df_sample, test_size=...)

In [ ]:
cnt = CountVectorizer(stop_words=None, tokenizer=tokenizer(), token_pattern=None, ngram_range=...)
cnt.fit(df_train["text"]);

In [ ]:
features = cnt.get_feature_names_out()
features

Cela fait beaucoup de caractéristiques ! C'est pour celà que l'on veut travailler avec des tableaux creux:

In [ ]:
# Entraîner un MultinomialNB sur notre set d'entrainement.
# On voudra le régler pour ne pas apprendre de probabilité à priori (indice : utiliser les arguments de la méthode)
# On voudra également fixer la probabilité à priori de chaque classe (indice : idem ). 
# [0.5,0.5] convient mais vous pouver essayer d'autres valeurs

clf = ...
x_train = cnt.transform(df_train["text"])
y_train = df_train["label"].to_numpy()
clf.fit(x_train, y_train);

In [ ]:
# On regarde les prédictions de notre modèle sur notre set de test.
# On utilise pour ça la méthode predict_proba()

x_test = cnt.transform(df_test["text"])
y_test = df_test["label"].to_numpy()
p_hat = clf.predict_proba(...)[:,1]

In [ ]:
# On veut utiliser une la méthode from_predictions() de la classe RocCurveDisplay.
# Mettez les bons arguments dans la méthode

RocCurveDisplay....(...);

Matrice de confusion avec un seuil de 0.5:

In [ ]:
# On veut utiliser une la méthode from_predictions() de la classe ConfusionMatrixDisplay.

ConfusionMatrixDisplay....(y_test, np.where(p_hat>=0.5, 1, 0));

### Modèle avec 5000 données et des n-grammes

On ne gardera qu'un échantillon de 5000 revues pour l'apprentissage et le test du modèle.

In [ ]:
df_sample = df.sample(..., random_state=14)

In [ ]:
df_train, df_test = train_test_split(df_sample, test_size=...)

In [ ]:
# Pour extraire les 1-gralles jusqu'au n-grammes, il faut ajouter en paramètre ngram_range=(1,n).
# Ici, on veut n=2

cnt = CountVectorizer(stop_words=None, tokenizer=tokenizer(), token_pattern=None, ngram_range=...)
cnt.fit(df_train["text"]);

In [ ]:
features = cnt.get_feature_names_out()
features

Comme le nombre de caractéristiques a augmenté très fortement, le calcul prend un certain temps:

In [ ]:
x_train = cnt.transform(df_train["text"])
y_train = df_train["label"].to_numpy()

In [ ]:
# Entraîner un MultinomialNB sur notre set d'entrainement.
# On voudra le régler pour ne pas apprendre de probabilité à priori (indice : utiliser les arguments de la méthode)
# On voudra également fixer la probabilité à priori de chaque classe (indice : idem ). 
# [0.5,0.5] convient mais vous pouver essayer d'autres valeurs

clf = ...
clf.fit(x_train, y_train);

In [ ]:
x_test = cnt.transform(df_test["text"])
y_test = df_test["label"].to_numpy()

In [ ]:
# On regarde les prédictions de notre modèle sur notre set de test.
# On utilise pour ça la méthode predict_proba()

p_hat = clf....(x_test)[:,1]

In [ ]:
# On veut utiliser une la méthode from_predictions() de la classe RocCurveDisplay.
# Mettez les bons arguments dans la méthode

RocCurveDisplay....(...);

Matrice de confusion avec un seuil à 0.5:

In [ ]:
# On veut utiliser une la méthode from_predictions() de la classe ConfusionMatrixDisplay.

ConfusionMatrixDisplay....(y_test, np.where(p_hat>=0.5, 1, 0));

On peut calculer la précision, le rappel ainsi que les taux des vrais positifs et faux positifs pour ce même seuil:

In [ ]:
precision_score(..., np.where(p_hat>=0.5, 1, 0))

In [ ]:
recall_score(..., np.where(p_hat>=0.5, 1, 0))

In [ ]:
C = confusion_matrix(..., np.where(p_hat>=0.5, 1, 0))
C

In [ ]:
# Calculez le taux de vrai positif

VP = ... # Vrai positif
FN = ... # Faux négatif
TVP = ...  # Le taux de vrai positif est aussi le rappel
TVP

In [ ]:
# Calculez le taux de faux positif
FP = ... # Faux positif
VN = ... # Vrai négatif
TFP = FP / (FP + VN)
TFP

In [ ]:
# On veut utiliser une la méthode from_predictions() de la classe RocCurveDisplay.
# Mettez les bons arguments dans la méthode

RocCurveDisplay....(...);
plt.gca().scatter([TFP], [TVP], color="orange", zorder=10, marker="x", s=50)
plt.gca().vlines(TFP, 0., ymax=TVP, ls=":", color="orange");
plt.gca().hlines(TVP, 0., xmax=TFP, ls=":", color="orange");

### On introduit TF-IDF !

Le simple comptage de most qu'on utilise jusqu'à présent donne trop d'importance aux mots fréquents. Nous allons introduire une variante un peu plus avancée. TF-IDF veut dire term frequency, inverse document frequency. Il combine deux vues:

- il va "pénaliser" des termes qui apparaissent très régulièrement et qui pourrait donc avoir un faible pouvoir prédictif. C'est le inverse document frequecy. Inversement, il "favorise" les termes peu fréquemment utilisés dans l'ensemble des documents.
- il va pénaliser les termes très utilisés au sein d'une même revue. Cela permet d'identifier des mots-vides que l'on n'aurait pas traité jusqu'à maintenant.

Dans les formules et algorithmes vuent jusqu'à présent, on utilise le term frequency $tf_j^{(i)}$ qui représente la fréquence d'apparition du mot $m_j$ dans le document textuel $t^{(i)}$ (ici, une review).

Sous forme de fonction indicatrice, cela donne :

$$tf_j^{(i)}=\sum\limits_{word\in t^{(i)}}1(word=m_j)$$

avec la fonction $1(u)$ qui vaut 1 si $u$ est vrai, et 0 sinon.

Le score $tf-idf_j^{(i)}$ pour un mot $m_j$ dans un document $t^{(i)}$ devient :

$$tf\text{-}idf_j^{(i)}=tf_j^{(i)}*ln(\frac{n}{n_{m_j}})$$

avec $n$ le nombre de document total et $n_{m_j}$ le nombre de document contenant le mot $m_j$.

On voit que plus un mot est rare, plus son score TF-IDJ va augmenter.

In [ ]:
# Transformer les données d'entrainement avec TfidfTransformer().

tfidf = ...
x_train = tfidf.fit_transform(x_train)

In [ ]:
# Entraîner un MultinomialNB sur notre set d'entrainement.
# On voudra le régler pour ne pas apprendre de probabilité à priori (indice : utiliser les arguments de la méthode)
# On voudra également fixer la probabilité à priori de chaque classe (indice : idem ). 
# [0.5,0.5] convient mais vous pouver essayer d'autres valeurs

clf = ...
clf.fit(x_train, y_train);

In [ ]:
# Transformer les données d'entrainement avec TfidfTransformer().

x_test = tfidf....(...)

In [ ]:
# On regarde les prédictions de notre modèle sur notre set de test.
# On utilise pour ça la méthode predict_proba()

p_hat = clf....(x_test)[:,1]

In [ ]:
# On veut utiliser une la méthode from_predictions() de la classe ConfusionMatrixDisplay.

ConfusionMatrixDisplay....(y_test, np.where(p_hat>=0.5, 1, 0));

In [ ]:
precision_score(..., np.where(p_hat>=0.5, 1, 0))

In [ ]:
recall_score(..., np.where(p_hat>=0.5, 1, 0))

In [ ]:
C = confusion_matrix(..., np.where(p_hat>=0.5, 1, 0))
C

In [ ]:
# Calculez le taux de vrai positif

VP = ... # Vrai positif
FN = ... # Faux négatif
TVP = ...  # Le taux de vrai positif est aussi le rappel
TVP

In [ ]:
# Calculez le taux de fuax positif
FP = ... # Faux positif
VN = ... # Vrai négatif
TFP = FP / (FP + VN)
TFP

In [ ]:
# On veut utiliser une la méthode from_predictions() de la classe RocCurveDisplay.
# Mettez les bons arguments dans la méthode

RocCurveDisplay....(...);
plt.gca().scatter([TFP], [TVP], color="orange", zorder=10, marker="x", s=50)
plt.gca().vlines(TFP, 0., ymax=TVP, ls=":", color="orange");
plt.gca().hlines(TVP, 0., xmax=TFP, ls=":", color="orange");

### Analyse du modèle

Quelques astuces pour identifier les n-grammes les plus significatifs sur la prise de décision de l'algorithme:

In [ ]:
# Utiliser l'attribut feature_count_[c] pour compter le nombre de fois où chaque n-gramme est associé à la classe c

negative_array = clf....[...] 
positive_array = clf....[...] 

In [ ]:
ratio_pos_neg = ... / (... + 1)
ratio_pos_neg.max()

In [ ]:
# Ne gardons que les features dont le ratio pos_neg est supérieur à 2.0

features[... >= ...]

In [ ]:
ratio_neg_pos = negative_array / (positive_array + 1)
ratio_neg_pos.max()

In [ ]:
# Ne gardons que les features dont le ratio neg_pos est supérieur à 2.0

features[... >= ...]

Avec un petit word cloud:

In [ ]:
wrd = wordcloud.WordCloud().generate_from_frequencies({f: r for f, r in zip(features, ratio_pos_neg)})
plt.imshow(wrd);

In [ ]:
wrd = wordcloud.WordCloud().generate_from_frequencies({f: r for f, r in zip(features, ratio_neg_pos)})
plt.imshow(wrd);

### Tests manuels !

In [ ]:
# Créer votre review et tester votre modèle dessus

def calculer_probabilite(review):
    # On tokenize
    x_test = tfidf.transform(cnt.transform([review]))
    # On calcule les probabilités
    return clf.predict_proba(x_test)[:,1]

my_review = ...

calculer_probabilite(my_review)